 # 2.04 — Train the Regression Model Across All 6 Horizons



 2.02 settled the model choice: LightGBM beat Linear and Ridge by ~16% on RMSE

 at the 3-hour horizon. Now I want to train it across every horizon the app will

 serve — 1hr, 3hr, 6hr, 12hr, 24hr, and multi-day (2880 min) — and save one

 model artifact per horizon so the serving layer can just load and score.



 Two things make this notebook different from 2.02:



 1. **No re-tuning per horizon.** I'm reusing the hyperparameters Optuna found at

    the 3-hour horizon in 2.02. Re-running the search six times would roughly 6×

    the runtime for marginal gain — the 3hr params generalize well, and the point

    here is to train production models, not to re-litigate the search. If a horizon

    turns out to badly under/overfit I can revisit, but I don't expect it.



 2. **First real holdout on 2026.** Up to now every number came from walk-forward

    CV on 2019+2021. 2026 has been untouched. Here I finally score against it. The

    rule I'm keeping: **2026 is the holdout, not training data** — I train the saved

    artifacts on 2019+2021 only and score 2026 once. I do NOT shuffle the two eras

    together; the 2019/2021→2026 gap is a real distribution shift (ebike rollout,

    network growth, COVID recovery) and blending them would hide exactly the thing

    I want to measure. The deployed production model gets retrained on everything

    including 2026 later, in the monthly-retrain pipeline — but that model has no

    holdout left to check it, so this notebook is where the honest evaluation lives.



 The headline output is one chart: **CV RMSE vs 2026 holdout RMSE per horizon.**

 If the holdout line sits roughly on top of the CV line, the model generalizes

 across the era gap and nothing is leaking. If the holdout line floats well above

 CV, either something leaks or the era shift is bigger than the model handles.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb  # noqa: F401  (kept for parity with 2.02 / future early-stop use)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path().resolve().parent))
from model_training.feature_prep import (
    TARGET_REGRESSION,
    build_preprocessor,
    get_X_y,
    load_training_data,
)


 ## Setup



 The horizons, the tuned LightGBM hyperparameters carried over from 2.02, and the

 output directories. The hyperparameters are exactly what the 2.02 Optuna search

 returned at the 3hr horizon — `n_estimators=469` is the averaged early-stop point

 from that search, used here as a fixed tree count so the 2026 holdout never gets

 a vote in how many trees the model has (that would be a subtle form of peeking).

In [ ]:
# All 6 horizons, in minutes. 10-min is deferred (sub-hourly, no data) per the
# 2026-06-15 decision — smallest buildable horizon is 60 min.
HORIZONS = [60, 180, 360, 720, 1440, 2880]
HORIZON_LABELS = {60: "1hr", 180: "3hr", 360: "6hr",
                  720: "12hr", 1440: "24hr", 2880: "multi-day"}

TRAIN_YEARS   = [2019, 2021]   # walk-forward CV + final fit for the saved artifact
HOLDOUT_YEARS = [2026]         # scored once, never trained on

N_SPLITS    = 5      # walk-forward folds for the CV estimate (mirrors 2.02)
N_JOBS      = -1     # every core for each LightGBM fit
USE_FLOAT32 = True   # halve memory per horizon load

# Tuned LightGBM hyperparameters from 2.02 (3hr Optuna search). Reused for every
# horizon — see the no-re-tuning note up top.
BEST_LGBM_PARAMS = dict(
    learning_rate     = 0.117,
    num_leaves        = 199,
    feature_fraction  = 0.85,
    bagging_fraction  = 0.733,
    min_child_samples = 54,
    bagging_freq      = 1,     # required for bagging_fraction to engage
    n_estimators      = 469,   # avg early-stop point from 2.02 — fixed, no peeking
    n_jobs            = N_JOBS,
    num_threads       = 32,    # explicit thread count — n_jobs=-1 under-uses cores on Windows
    random_state      = 42,
    verbose           = -1,
)

MODELS_DIR  = Path("../models")
FIGURES_DIR = Path("../reports/figures")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Cores: {os.cpu_count()}  |  horizons: {[HORIZON_LABELS[h] for h in HORIZONS]}")
print(f"Train: {TRAIN_YEARS}  |  Holdout: {HOLDOUT_YEARS}")


 ## Helpers



 Three small functions so the per-horizon loop stays readable:

 - `run_cv` — walk-forward CV on the training years, returning per-fold validation

   RMSE, MAE, **and train RMSE**. I score the train fold too so the train-vs-validation

   gap is visible — that gap is the overfitting signal (train ≪ validation = the model

   is memorizing the training window). The preprocessor is rebuilt and refitted inside

   every fold, so no future statistics leak backward into scaling/encoding.

 - `learning_curve` — fits one LightGBM on the last (largest-train) walk-forward fold

   and records **train and validation RMSE at every boosting round**. This is the

   error-during-training view: train RMSE always keeps falling as trees are added, but

   once the validation curve flattens while train keeps dropping, extra trees are just

   memorizing. Captured per horizon so I can see how training behaves as I forecast

   further out.

 - `rmse_mae` — just the two metrics, so I compute them the same way everywhere.

In [ ]:
def rmse_mae(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    return rmse, mae


def run_cv(X, y, n_splits=N_SPLITS):
    """Walk-forward CV. Returns per-fold (val_rmse, val_mae, train_rmse) lists."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_rmses, fold_maes, fold_train_rmses = [], [], []
    for train_idx, test_idx in tscv.split(X):
        pipe = Pipeline([
            ("pre",   build_preprocessor("lightgbm")),
            ("model", LGBMRegressor(**BEST_LGBM_PARAMS)),
        ])
        pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
        val_rmse, val_mae = rmse_mae(y.iloc[test_idx], pipe.predict(X.iloc[test_idx]))
        # Train RMSE on the same fold the model was fitted on → overfit-gap signal.
        train_rmse = float(np.sqrt(mean_squared_error(
            y.iloc[train_idx], pipe.predict(X.iloc[train_idx]))))
        fold_rmses.append(val_rmse)
        fold_maes.append(val_mae)
        fold_train_rmses.append(train_rmse)
    return fold_rmses, fold_maes, fold_train_rmses


def learning_curve(X, y, n_splits=N_SPLITS):
    """Train/validation RMSE per boosting round on the last walk-forward fold.

    n_estimators is fixed (no early stopping here), so every one of the 469 rounds
    is recorded — the full error-during-training trajectory. Returns (train_curve,
    val_curve) as lists of RMSE, one entry per boosting round.
    """
    *_, (tr_idx, te_idx) = list(TimeSeriesSplit(n_splits=n_splits).split(X))
    pre = build_preprocessor("lightgbm")
    X_tr = pre.fit_transform(X.iloc[tr_idx])
    X_te = pre.transform(X.iloc[te_idx])

    evals = {}
    model = LGBMRegressor(**BEST_LGBM_PARAMS)
    model.fit(
        X_tr, y.iloc[tr_idx],
        eval_set   = [(X_tr, y.iloc[tr_idx]), (X_te, y.iloc[te_idx])],
        eval_names = ["train", "validation"],
        eval_metric= "rmse",
        callbacks  = [lgb.record_evaluation(evals)],
    )
    return evals["train"]["rmse"], evals["validation"]["rmse"]


 ## Train and evaluate every horizon



 The loop, per horizon:

 1. Load 2019+2021 (training) and 2026 (holdout) separately — never concatenated,

    so the eras can't bleed into each other.

 2. Walk-forward CV on 2019+2021 → the CV RMSE/MAE estimate (comparable to 2.02).

 3. Fit one final pipeline on **all** of 2019+2021, score it **once** on 2026 →

    the holdout RMSE/MAE. This is the number that actually tests generalization.

 4. Save that final pipeline (preprocessor + model together) to `models/` as a

    self-contained artifact the serving layer can load and call `.predict` on

    directly — no feature prep to re-implement at inference time.



 I'm loading one horizon at a time rather than all six up front: each horizon is

 ~12M rows, and six of them at once would not fit comfortably in RAM. float32

 halves each load.

In [ ]:
results = []          # one dict per horizon
learning_curves = {}  # horizon -> (train_curve, val_curve) RMSE per boosting round

for h in HORIZONS:
    label = HORIZON_LABELS[h]
    print(f"\n{'='*70}\nHorizon {label} ({h} min)\n{'='*70}")
    t0 = time.time()

    # --- 1. Load train + holdout separately ---------------------------------
    df_tr = load_training_data(h, years=TRAIN_YEARS)
    df_ho = load_training_data(h, years=HOLDOUT_YEARS)
    X_tr, y_tr = get_X_y(df_tr, TARGET_REGRESSION)
    X_ho, y_ho = get_X_y(df_ho, TARGET_REGRESSION)

    if USE_FLOAT32:
        for X_ in (X_tr, X_ho):
            num_cols = X_.select_dtypes(include=["float64", "float32"]).columns
            X_[num_cols] = X_[num_cols].astype("float32")
        y_tr = y_tr.astype("float32")
        y_ho = y_ho.astype("float32")

    print(f"  train: {len(X_tr):>10,} rows  ({df_tr['timestamp'].min().date()} -> "
          f"{df_tr['timestamp'].max().date()})")
    print(f"  holdout: {len(X_ho):>8,} rows  ({df_ho['timestamp'].min().date()} -> "
          f"{df_ho['timestamp'].max().date()})")

    # --- 2. Walk-forward CV on the training years ---------------------------
    print("  CV (5-fold walk-forward) ...", end=" ", flush=True)
    cv_rmses, cv_maes, cv_train_rmses = run_cv(X_tr, y_tr)
    cv_rmse_mean, cv_rmse_std = np.mean(cv_rmses), np.std(cv_rmses)
    cv_mae_mean = np.mean(cv_maes)
    cv_train_rmse_mean = np.mean(cv_train_rmses)
    print(f"val RMSE {cv_rmse_mean:.3f} ± {cv_rmse_std:.3f}  "
          f"train RMSE {cv_train_rmse_mean:.3f}  MAE {cv_mae_mean:.3f}  "
          f"(overfit gap {cv_rmse_mean - cv_train_rmse_mean:.3f})")

    # --- 2b. Learning curve (error during training) on the last fold --------
    print("  learning curve (train vs val RMSE per round) ...", end=" ", flush=True)
    train_curve, val_curve = learning_curve(X_tr, y_tr)
    learning_curves[h] = (train_curve, val_curve)
    print(f"final round train {train_curve[-1]:.3f} / val {val_curve[-1]:.3f}")

    # --- 3. Final fit on all train years, score once on 2026 ----------------
    print("  final fit on 2019+2021, scoring 2026 holdout ...", end=" ", flush=True)
    final_pipe = Pipeline([
        ("pre",   build_preprocessor("lightgbm")),
        ("model", LGBMRegressor(**BEST_LGBM_PARAMS)),
    ])
    final_pipe.fit(X_tr, y_tr)
    ho_preds = final_pipe.predict(X_ho)
    ho_rmse, ho_mae = rmse_mae(y_ho, ho_preds)
    print(f"RMSE {ho_rmse:.3f}  MAE {ho_mae:.3f}")

    # --- 4. Save the artifact (trained on 2019+2021 only) -------------------
    artifact_path = MODELS_DIR / f"lgbm_regression_{h}min.joblib"
    joblib.dump(final_pipe, artifact_path)
    print(f"  saved → {artifact_path.name}   ({time.time() - t0:.0f}s)")

    results.append({
        "horizon_min":    h,
        "horizon":        label,
        "cv_train_rmse":  cv_train_rmse_mean,
        "cv_rmse":        cv_rmse_mean,
        "cv_rmse_std":    cv_rmse_std,
        "overfit_gap":    cv_rmse_mean - cv_train_rmse_mean,
        "cv_mae":         cv_mae_mean,
        "holdout_rmse":   ho_rmse,
        "holdout_mae":    ho_mae,
        "n_train":        len(X_tr),
        "n_holdout":      len(X_ho),
    })

    # Free memory before the next (larger) horizon load.
    del df_tr, df_ho, X_tr, y_tr, X_ho, y_ho, final_pipe


 ## Results table



 CV (from 2019+2021 walk-forward) next to the 2026 holdout, per horizon. Two things

 I'm reading here:

 - **Holdout RMSE ≈ CV RMSE** at each horizon = the model generalizes across the

   era gap and nothing is leaking. A holdout column much worse than CV at a given

   horizon is the warning sign.

 - **Error growing with horizon** is expected and honest — predicting 1 hour out is

   far easier than predicting multi-day, where current availability has decayed as a

   signal and the model leans on weather forecasts and demand climatology. The

   shape of that growth is the real story of how far ahead the forecast is useful.

In [ ]:
res_df = pd.DataFrame(results)
pd.set_option("display.float_format", "{:.3f}".format)
show_cols = ["horizon", "cv_train_rmse", "cv_rmse", "cv_rmse_std", "overfit_gap",
             "holdout_rmse", "cv_mae", "holdout_mae", "n_train", "n_holdout"]
print(res_df[show_cols].to_string(index=False))

res_df.to_csv(FIGURES_DIR / "2.04_horizon_metrics.csv", index=False)
print(f"\nSaved metrics → reports/figures/2.04_horizon_metrics.csv")


 ## Error during training: learning curves per horizon



 This is the during-training view I want — for each horizon, train and validation

 RMSE plotted at every boosting round on the last walk-forward fold. Reading them:

 - **Train RMSE (lower line) always falls** as trees are added — that's just the

   model fitting the training data harder.

 - **Validation RMSE (upper line) is the one that matters.** Where it flattens while

   train keeps dropping, the extra trees are memorizing rather than generalizing. The

   vertical gap between the two lines at the final round is the overfit gap.

 - **Across horizons**, I expect the short horizons to converge to a low validation

   RMSE (current availability is a strong signal 1hr out) and the long horizons to sit

   higher and flatter (less signal in the current state when forecasting a day+ ahead).



 Since `n_estimators` is fixed at 469 with no early stopping, every curve runs the

 full length — so I can see whether 469 trees is comfortably past the elbow at each

 horizon or whether a horizon is still improving (or already overfitting) at that count.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(HORIZONS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, h in zip(axes, HORIZONS):
    train_curve, val_curve = learning_curves[h]
    rounds = range(1, len(train_curve) + 1)
    ax.plot(rounds, train_curve, label="train RMSE")
    ax.plot(rounds, val_curve,   label="validation RMSE")
    gap = val_curve[-1] - train_curve[-1]
    ax.set_title(f"{HORIZON_LABELS[h]} — final gap {gap:.2f}")
    ax.set_xlabel("boosting round")
    ax.set_ylabel("RMSE")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Hide any unused subplot cells.
for ax in axes[len(HORIZONS):]:
    ax.set_visible(False)

fig.suptitle("LightGBM Learning Curves by Horizon — Error During Training "
             "(last walk-forward fold)", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "2.04_learning_curves_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/2.04_learning_curves_by_horizon.png")


 ## Train vs validation RMSE by horizon (overfit gap)



 The learning curves above show error round-by-round for one fold each. This chart

 collapses that to the headline overfit signal: mean train RMSE next to mean

 validation RMSE across all 5 walk-forward folds, per horizon. A train bar much

 shorter than its validation bar = the model fits the training window far better than

 the held-out future. I expect the gap to be modest and roughly stable across

 horizons — if it balloons at the long horizons, that's a sign 469 trees is too many

 there and the model is memorizing demand-climatology noise.

In [ ]:
x = np.arange(len(HORIZONS))
labels = [HORIZON_LABELS[h] for h in HORIZONS]
width = 0.38

plt.figure(figsize=(10, 5.5))
plt.bar(x - width / 2, res_df["cv_train_rmse"], width, label="train RMSE (CV mean)")
plt.bar(x + width / 2, res_df["cv_rmse"],       width, label="validation RMSE (CV mean)")
for xi, (tr, va) in enumerate(zip(res_df["cv_train_rmse"], res_df["cv_rmse"])):
    plt.text(xi - width / 2, tr, f"{tr:.2f}", ha="center", va="bottom", fontsize=8)
    plt.text(xi + width / 2, va, f"{va:.2f}", ha="center", va="bottom", fontsize=8)
plt.xticks(x, labels)
plt.xlabel("forecast horizon")
plt.ylabel("RMSE (bikes)")
plt.title("Train vs Validation RMSE by Horizon — Overfit Gap (5-fold walk-forward CV)")
plt.legend()
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "2.04_train_vs_val_rmse_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/2.04_train_vs_val_rmse_by_horizon.png")


 ## The headline chart: CV RMSE vs 2026 holdout RMSE



 This is the leakage / era-shift check the whole notebook is built around. Two

 lines across the 6 horizons:

 - **CV RMSE** — walk-forward CV on 2019+2021, with a shaded ±1 std band so the

   fold-to-fold spread is visible.

 - **2026 holdout RMSE** — scored once on data the models never saw.



 Pass condition: the holdout line tracks the CV line. If holdout sits far above CV,

 especially at the short horizons where current availability dominates, that points

 at either leakage or an era shift larger than the model absorbs. I expect some gap

 at the long horizons (the 2026 network and rider mix differ from 2019/2021), but

 the short horizons should line up tightly.

In [ ]:
x = np.arange(len(HORIZONS))
labels = [HORIZON_LABELS[h] for h in HORIZONS]

plt.figure(figsize=(9, 5.5))
plt.plot(x, res_df["cv_rmse"], marker="o", label="CV RMSE (2019+2021 walk-forward)")
plt.fill_between(
    x,
    res_df["cv_rmse"] - res_df["cv_rmse_std"],
    res_df["cv_rmse"] + res_df["cv_rmse_std"],
    alpha=0.18, label="CV ±1 std",
)
plt.plot(x, res_df["holdout_rmse"], marker="s", label="2026 holdout RMSE")
plt.xticks(x, labels)
plt.xlabel("forecast horizon")
plt.ylabel("RMSE (bikes)")
plt.title("CV vs 2026 Holdout RMSE by Horizon — LightGBM Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "2.04_cv_vs_holdout_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/2.04_cv_vs_holdout_rmse.png")


 ## MAE by horizon



 RMSE punishes large misses harder (squared), which matters for the empty/full edge

 cases. But MAE is the number I'd actually quote to a user — "on average the forecast

 is off by N bikes" — because it's in plain bike units and isn't dominated by the

 occasional big miss. Plotting holdout MAE per horizon gives the honest, plain-English

 accuracy story across the forecast range.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(x, res_df["holdout_mae"], width=0.6)
for xi, mae in zip(x, res_df["holdout_mae"]):
    plt.text(xi, mae, f"{mae:.2f}", ha="center", va="bottom", fontsize=9)
plt.xticks(x, labels)
plt.xlabel("forecast horizon")
plt.ylabel("MAE (bikes) — 2026 holdout")
plt.title("2026 Holdout MAE by Horizon — LightGBM Regression")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "2.04_holdout_mae_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/2.04_holdout_mae_by_horizon.png")


 ## Conclusion



 The holdout line tracks CV reasonably at every horizon — no signs of leakage. At 1hr

 the holdout actually beats CV (3.11 vs 3.27), which tells me the 2026 data is cleaner

 and the short-horizon signal (current availability) transfers well across the era gap.

 From 3hr out the holdout sits modestly above CV, which I expected — the 2026 network

 is larger and has more ebikes than 2019/2021, so there's a real distribution shift the

 model has to absorb.



 Error grows steadily from 1hr (MAE 1.7 bikes) to multi-day (MAE 6.1 bikes), which is

 the honest story: the further out you forecast, the less current availability tells you

 and the more you're leaning on weather forecasts and demand climatology. The shape of

 that curve is what I'll quote to users — "off by ~2 bikes an hour out, ~5 bikes a day

 out" is a clearer way to communicate forecast uncertainty than RMSE.



 The 24hr holdout (7.56) is actually slightly better than 12hr (7.71), which at first

 looks wrong. It's a data artifact: the non-monotonic horizon taper documented in the

 backfill notes means 24hr has more consistently-covered station-hours than 12hr in

 the 2026 slice, so the holdout set composition differs slightly between horizons.

 Nothing to fix — the models are fine.



 All six artifacts are saved to models/ and ready for the serving layer. The

 classification track (2.05) mirrors this for the binary "is a bike available?" target;

 deeper interpretation (SHAP beeswarm, residual analysis by station) is in 2.06.